In [6]:
!pip install s3fs
!pip install openpyxl

  Using cached fsspec-2024.9.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2024.9.0-py3-none-any.whl (179 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.6.0
ERROR: Cannot uninstall fsspec 2023.6.0, RECORD file not found. You might be able to recover from this via: 'pip install --force-reinstall --no-deps fsspec==2023.6.0'.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 23.6 MB/s eta 0:00:00


In [7]:
import boto3
import pandas as pd
import numpy as np

# Preprocess the dataset
def preprocess_data(file_path):  
    # Read Excel file instead of CSV
    df = pd.read_excel(file_path)  # Change this to pd.read_excel for .xlsx file
    # Convert to datetime columns
    df["firstorder"] = pd.to_datetime(df["firstorder"], errors='coerce')
    df["lastorder"] = pd.to_datetime(df["lastorder"], errors='coerce')
    # Drop Rows with null values
    df = df.dropna()    
    # Create Column which gives the days between the last order and the first order
    df["first_last_days_diff"] = (df['lastorder'] - df['firstorder']).dt.days
    # Create Column which gives the days between when the customer record was created and the first order
    df['created'] = pd.to_datetime(df['created'])
    df['created_first_days_diff'] = (df['created'] - df['firstorder']).dt.days
    # Drop Columns
    df.drop(['custid', 'created', 'firstorder', 'lastorder'], axis=1, inplace=True)
    # Apply one hot encoding on favday and city columns
    df = pd.get_dummies(df, prefix=['favday', 'city'], columns=['favday', 'city'])
    return df

# Set the required configurations
model_name = "churn_model"
env = "dev"
# S3 Bucket
default_bucket = "sagemaker-us-east-1-992382734211"
# Preprocess the dataset
storedata = preprocess_data(f"s3://{default_bucket}/data/storedata_total.xlsx")  # Updated to .xlsx

/opt/conda/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [8]:
# Function to split the dataset into training, validation, and test sets
def split_datasets(df):
    y = df.pop("retained")  # Assuming 'retained' is the target column
    X_pre = df
    y_pre = y.to_numpy().reshape(len(y), 1)
    feature_names = list(X_pre.columns)
    X = np.concatenate((y_pre, X_pre), axis=1)
    np.random.shuffle(X)
    # Splitting the dataset: 70% training, 15% validation, 15% testing
    train, validation, test = np.split(X, [int(.7 * len(X)), int(.85 * len(X))])
    return feature_names, train, validation, test

# Split the preprocessed dataset
feature_names, train, validation, test = split_datasets(storedata)

# Save train, validation, and test datasets to their respective S3 paths
pd.DataFrame(train).to_csv(f"s3://{default_bucket}/data/train/train.csv", header=False, index=False)
pd.DataFrame(validation).to_csv(f"s3://{default_bucket}/data/validation/validation.csv", header=False, index=False)
pd.DataFrame(test).to_csv(f"s3://{default_bucket}/data/test/test.csv", header=False, index=False)


In [9]:
import boto3
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import ContinuousParameter, IntegerParameter, HyperparameterTuner

# Set up the SageMaker session and the region
region = boto3.Session().region_name  # Ensure this matches your AWS region, e.g., 'us-east-1'
role = sagemaker.get_execution_role()  # Gets the IAM role used by SageMaker

# Set the default S3 bucket and data paths
default_bucket = "sagemaker-us-east-1-992382734211"

# Define the training and validation input data from S3
s3_input_train = TrainingInput(
    s3_data=f"s3://{default_bucket}/data/train/", content_type="csv"
)
s3_input_validation = TrainingInput(
    s3_data=f"s3://{default_bucket}/data/validation/", content_type="csv"
)

# Set the fixed hyperparameters for the XGBoost algorithm
fixed_hyperparameters = {
    "eval_metric": "auc",
    "objective": "binary:logistic",
    "num_round": "100",
    "rate_drop": "0.3",
    "tweedie_variance_power": "1.4"
}

# Retrieve the XGBoost container image URI for your region
container = sagemaker.image_uris.retrieve("xgboost", region, version="0.90-2")

# Create the SageMaker Estimator for the XGBoost algorithm
estimator = sagemaker.estimator.Estimator(
    container,
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",  # Feel free to adjust based on your needs
    hyperparameters=fixed_hyperparameters,
    output_path=f"s3://{default_bucket}/output",
    sagemaker_session=sagemaker.Session()  # Ensure a session is established
)

# Define the hyperparameter tuning ranges
hyperparameter_ranges = {
    "eta": ContinuousParameter(0, 1),
    "min_child_weight": ContinuousParameter(1, 10),
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
}

# Set the objective metric for tuning
objective_metric_name = "validation:auc"

# Set up the Hyperparameter Tuner
tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name=objective_metric_name,
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,  # Total number of jobs to run
    max_parallel_jobs=2  # Number of jobs to run in parallel
)

# Start the tuning job
tuner.fit({
    "train": s3_input_train,
    "validation": s3_input_validation
}, include_cls_metadata=False)

# Monitor the progress of the tuning job and retrieve results
# Describe the hyperparameter tuning job
tuning_job_result = boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name
)

# Output how many training jobs have completed
job_count = tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print(f"{job_count} training jobs have completed")

# Check if a best model was found
from pprint import pprint
if tuning_job_result.get("BestTrainingJob", None):
    print("Best Model found so far:")
    pprint(tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


........................................................................!
10 training jobs have completed
Best Model found so far:
{'CreationTime': datetime.datetime(2024, 9, 9, 1, 44, 52, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:auc',
                                                 'Value': 0.9758859872817993},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2024, 9, 9, 1, 45, 29, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:992382734211:training-job/sagemaker-xgboost-240909-0140-008-d190cac0',
 'TrainingJobName': 'sagemaker-xgboost-240909-0140-008-d190cac0',
 'TrainingJobStatus': 'Completed',
 'TrainingStartTime': datetime.datetime(2024, 9, 9, 1, 44, 55, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '1.947913919722267',
                          'eta': '0.12693746328556488',
                          'max_depth': '5',
                          'min_child_weight': '7.798248811435